<a href="https://colab.research.google.com/github/pablojbec/E-commerce-Funnel-Cohort-Analysis/blob/main/Analisis_funnel_cohortesSQL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Introducción

Este estudio busca comprender el comportamiento de los usuarios en la plataforma de e-commerce para detectar en qué etapa del proceso de compra se genera el mayor abandono y la retención de usuarios para entender si regresan después de registrarse.

# 1. Carga y exploración de datasets

In [1]:
#Librerias
import pandas as pd

In [2]:
from google.colab import drive
drive.mount('/content/drive')

customers = pd.read_csv("/content/drive/MyDrive/Datasets/5. Ecommerce cohortes/customers.csv")
events = pd.read_csv("/content/drive/MyDrive/Datasets/5. Ecommerce cohortes/events.csv")
orders = pd.read_csv("/content/drive/MyDrive/Datasets/5. Ecommerce cohortes/orders.csv")
sessions = pd.read_csv("/content/drive/MyDrive/Datasets/5. Ecommerce cohortes/sessions.csv")


print(customers.head(10))
print(events.head(10))
print(orders.head(10))
print(sessions.head(10))

Mounted at /content/drive
   customer_id              name                       email country  age  \
0            1  Jennifer Salinas      nicholas59@example.org      JP   71   
1            2     Phillip Ramos  christinarubio@example.com      IN   26   
2            3       Dawn Fowler       jessica03@example.org      BR   21   
3            4      Mario Butler         paula27@example.org      FR   63   
4            5       Amber Brown         kevin85@example.net      BR   19   
5            6     Kevin Chapman    michaelpoole@example.org      US   28   
6            7   Timothy Sanchez    shannonbrown@example.org      IN   48   
7            8   Rachael Morales         tammy08@example.com      GB   33   
8            9      Laura Guzman         awright@example.org      US   32   
9           10   Alexander Lopez    brianmaxwell@example.org      JP   39   

  signup_date  marketing_opt_in  
0  2020-09-04              True  
1  2020-04-05             False  
2  2023-08-31           

#  2. Limpieza y Exploración de datos


In [3]:
customers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   customer_id       20000 non-null  int64 
 1   name              20000 non-null  object
 2   email             20000 non-null  object
 3   country           20000 non-null  object
 4   age               20000 non-null  int64 
 5   signup_date       20000 non-null  object
 6   marketing_opt_in  20000 non-null  bool  
dtypes: bool(1), int64(2), object(4)
memory usage: 957.2+ KB


In [4]:
events.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 760958 entries, 0 to 760957
Data columns (total 10 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   event_id      760958 non-null  int64  
 1   session_id    760958 non-null  int64  
 2   timestamp     760958 non-null  object 
 3   event_type    760958 non-null  object 
 4   product_id    682469 non-null  float64
 5   qty           143126 non-null  float64
 6   cart_size     44909 non-null   float64
 7   payment       33580 non-null   object 
 8   discount_pct  33580 non-null   float64
 9   amount_usd    33580 non-null   float64
dtypes: float64(5), int64(2), object(3)
memory usage: 58.1+ MB


In [5]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33580 entries, 0 to 33579
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   order_id        33580 non-null  int64  
 1   customer_id     33580 non-null  int64  
 2   order_time      33580 non-null  object 
 3   payment_method  33580 non-null  object 
 4   discount_pct    33580 non-null  int64  
 5   subtotal_usd    33580 non-null  float64
 6   total_usd       33580 non-null  float64
 7   country         33580 non-null  object 
 8   device          33580 non-null  object 
 9   source          33580 non-null  object 
dtypes: float64(2), int64(3), object(5)
memory usage: 2.6+ MB


In [6]:
sessions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120000 entries, 0 to 119999
Data columns (total 6 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   session_id   120000 non-null  int64 
 1   customer_id  120000 non-null  int64 
 2   start_time   120000 non-null  object
 3   device       120000 non-null  object
 4   source       120000 non-null  object
 5   country      120000 non-null  object
dtypes: int64(2), object(4)
memory usage: 5.5+ MB


In [8]:
#Exploración de variables numéricas
customers_num=['customer_id','age']
print(customers[customers_num].describe())

events_num=['event_id', 'session_id', 'product_id', 'qty', 'cart_size', 'discount_pct', 'amount_usd']
print(events[events_num].describe())

orders_num=['order_id', 'customer_id', 'discount_pct', 'subtotal_usd', 'total_usd']
print(orders[orders_num].describe())

session_num=['session_id', 'customer_id']
print(sessions[session_num].describe())

        customer_id           age
count  20000.000000  20000.000000
mean   10000.500000     46.492550
std     5773.647028     16.767961
min        1.000000     18.000000
25%     5000.750000     32.000000
50%    10000.500000     46.500000
75%    15000.250000     61.000000
max    20000.000000     75.000000
            event_id     session_id     product_id            qty  \
count  760958.000000  760958.000000  682469.000000  143126.000000   
mean   380479.500000   60153.502012     652.313074       1.301748   
std    219669.797409   34613.117088     340.279129       0.783436   
min         1.000000       1.000000       1.000000       1.000000   
25%    190240.250000   30314.250000     368.000000       1.000000   
50%    380479.500000   60176.000000     674.000000       1.000000   
75%    570718.750000   90110.000000     954.000000       1.000000   
max    760958.000000  120000.000000    1197.000000       4.000000   

          cart_size  discount_pct    amount_usd  
count  44909.000000  3

## 2.1 Revisión de valores nulos

In [10]:
print(customers.isnull().sum())
print(customers.isnull().mean())
print("____________________________________")
print(events.isnull().sum())
print(events.isnull().mean())
print("____________________________________")
print(orders.isnull().sum())
print(orders.isnull().mean())
print("____________________________________")
print(sessions.isnull().sum())
print(sessions.isnull().mean())

customer_id         0
name                0
email               0
country             0
age                 0
signup_date         0
marketing_opt_in    0
dtype: int64
customer_id         0.0
name                0.0
email               0.0
country             0.0
age                 0.0
signup_date         0.0
marketing_opt_in    0.0
dtype: float64
____________________________________
event_id             0
session_id           0
timestamp            0
event_type           0
product_id       78489
qty             617832
cart_size       716049
payment         727378
discount_pct    727378
amount_usd      727378
dtype: int64
event_id        0.000000
session_id      0.000000
timestamp       0.000000
event_type      0.000000
product_id      0.103145
qty             0.811913
cart_size       0.940984
payment         0.955871
discount_pct    0.955871
amount_usd      0.955871
dtype: float64
____________________________________
order_id          0
customer_id       0
order_time        0
payment_

#### Comentario:
De acuerdo con lo anterior, se tiene que las columnas de las tablas `customers`, `orders` y `sessións` no tienen valores nulos.

La tabla `events` cuenta con valores nulos en las columnas de `product_id` (10%), `qty`(81%), `cart_size`(94%), `payment`(96%), `discount_pct`(96%), `amount_usd `(96%).

 Dado el objeto de análisis, se decide conservar los estos valores nulos.

## 2.2 Correción de tipos de datos

In [11]:
# customers
customers["signup_date"] = pd.to_datetime(customers["signup_date"])
customers["country"] = customers["country"].astype("category")
customers["name"] = customers["name"].astype("string")
customers["email"] = customers["email"].astype("string")

# events
events["timestamp"] = pd.to_datetime(events["timestamp"])
events["event_type"] = events["event_type"].astype("category")
events["payment"] = events["payment"].astype("category")
events["product_id"] = events["product_id"].astype("Int64")
events["qty"] = events["qty"].astype("Int64")
events["cart_size"] = events["cart_size"].astype("Int64")

# orders
orders["order_time"] = pd.to_datetime(orders["order_time"])
for col in ["payment_method", "country", "device", "source"]:
    orders[col] = orders[col].astype("category")

# sessions
sessions["start_time"] = pd.to_datetime(sessions["start_time"])
for col in ["device", "source", "country"]:
    sessions[col] = sessions[col].astype("category")

#### Comentario:
Se realizó la revisión de los tipos de datos con el objetivo de asegurar que cada variable estuviera representada de acuerdo con su naturaleza.
- Las columnas correspondientes a fechas y horas (signup_date, timestamp, order_time y start_time) se convierten al tipo datetime, lo que facilita el análisis temporal y las operaciones sobre fechas.
- Las variables categóricas, como country, device, source, event_type y los métodos de pago, se convierten al tipo category para optimizar el uso de memoria y mejorar la eficiencia en las operaciones de agrupación y filtrado.
- Los identificadores y cantidades que presentan valores nulos (product_id, qty y cart_size) se convierten al tipo entero nullable (Int64), preservando su naturaleza numérica sin perder la capacidad de representar valores faltantes.
- Las columnas de texto (name y email) se convierten al tipo string para un manejo más consistente de datos textuales en Pandas.

## 3. Análisis de funnel

In [12]:
import duckdb
duckdb.sql("""
   SELECT
    s.session_id,
    s.customer_id,
    s.start_time,
    s.device,
    s.source,
    s.country,
    e.event_id,
    e.timestamp,
    e.event_type,
    e.product_id,
    e.amount_usd
FROM sessions s
INNER JOIN events e
    ON s.session_id = e.session_id;
""")

┌────────────┬─────────────┬─────────────────────┬─────────────────────────────────────┬──────────────────────────────────────────────────────────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────┬──────────┬─────────────────────┬──────────────────────────────────────────────────────────┬────────────┬────────────┐
│ session_id │ customer_id │     start_time      │               device                │                              source                              │                                                  country                                                   │ event_id │      timestamp      │                        event_type                        │ product_id │ amount_usd │
│   int64    │    int64    │    timestamp_ns     │ enum('desktop', 'mobile', 'tablet') │ enum('direct', 'email', 'organic', 'paid', 'referral', 'social') │ enum('ae', 'au', 'br', 'ca', 'de', 'es', 'fr', 'gb', 'in', 'jp', 'mx', 'nl', '

In [13]:
duckdb.sql("""
SELECT
    s.session_id,
    s.customer_id,
    s.source,
    s.device,
    e.timestamp,
    e.event_type
FROM sessions s
JOIN events e
    ON s.session_id = e.session_id
ORDER BY
    s.session_id,
    e.timestamp;
""")

┌────────────┬─────────────┬──────────────────────────────────────────────────────────────────┬─────────────────────────────────────┬─────────────────────┬──────────────────────────────────────────────────────────┐
│ session_id │ customer_id │                              source                              │               device                │      timestamp      │                        event_type                        │
│   int64    │    int64    │ enum('direct', 'email', 'organic', 'paid', 'referral', 'social') │ enum('desktop', 'mobile', 'tablet') │    timestamp_ns     │ enum('add_to_cart', 'checkout', 'page_view', 'purchase') │
├────────────┼─────────────┼──────────────────────────────────────────────────────────────────┼─────────────────────────────────────┼─────────────────────┼──────────────────────────────────────────────────────────┤
│          1 │       12360 │ email                                                            │ mobile                              │ 2021-1

In [14]:
duckdb.sql("""
SELECT
    event_type,
    COUNT(*) AS total
FROM events
GROUP BY event_type
ORDER BY total DESC;
""")

┌──────────────────────────────────────────────────────────┬────────┐
│                        event_type                        │ total  │
│ enum('add_to_cart', 'checkout', 'page_view', 'purchase') │ int64  │
├──────────────────────────────────────────────────────────┼────────┤
│ page_view                                                │ 539343 │
│ add_to_cart                                              │ 143126 │
│ checkout                                                 │  44909 │
│ purchase                                                 │  33580 │
└──────────────────────────────────────────────────────────┴────────┘

In [20]:
duckdb.sql("""
WITH base AS (
    SELECT
        s.session_id,
        s.customer_id,
        s.source,
        s.device,
        s.country,
        e.timestamp,
        e.event_type
    FROM sessions s
    INNER JOIN events e
        ON s.session_id = e.session_id
),

cte_page_view AS (
    SELECT DISTINCT customer_id
    FROM base
    WHERE event_type = 'page_view' ),
cte_add_to_cart AS (
    SELECT DISTINCT customer_id
    FROM base
    WHERE event_type  = 'add_to_cart' ),
cte_checkout AS (
    SELECT DISTINCT customer_id
    FROM base
    WHERE event_type = 'checkout' ),
cte_purchase AS (
    SELECT DISTINCT customer_id
    FROM base
    WHERE event_type = 'purchase' ),

baseline AS (
    SELECT
    (SELECT COUNT(*) FROM cte_page_view)      AS page_view_users,
    (SELECT COUNT(*) FROM cte_add_to_cart)    AS add_to_cart_users,
    (SELECT COUNT(*) FROM cte_checkout)       AS checkout_users,
    (SELECT COUNT(*) FROM cte_purchase)       AS purchase_users
)

SELECT *,
ROUND(add_to_cart_users * 100 / page_view_users, 2) AS pageview_addtocart_conv,
ROUND(checkout_users*100 / add_to_cart_users, 2) AS addtocart_checkout_conv,
ROUND(purchase_users*100/checkout_users, 2) AS checkout_purchase_conv,
FROM baseline

""")

┌─────────────────┬───────────────────┬────────────────┬────────────────┬─────────────────────────┬─────────────────────────┬────────────────────────┐
│ page_view_users │ add_to_cart_users │ checkout_users │ purchase_users │ pageview_addtocart_conv │ addtocart_checkout_conv │ checkout_purchase_conv │
│      int64      │       int64       │     int64      │     int64      │         double          │         double          │         double         │
├─────────────────┼───────────────────┼────────────────┼────────────────┼─────────────────────────┼─────────────────────────┼────────────────────────┤
│           19945 │             19660 │          17898 │          16268 │                   98.57 │                   91.04 │                  90.89 │
└─────────────────┴───────────────────┴────────────────┴────────────────┴─────────────────────────┴─────────────────────────┴────────────────────────┘

#### Comentario

- El análisis del funnel permitió identificar el comportamiento de los usuarios durante el proceso de compra y detectar las etapas donde ocurre la mayor pérdida de usuarios.
- Los resultados del análisis del funnel muestran que la mayoría de los usuarios avanzan correctamente desde la visita inicial de la página hasta la compra, con una conversión final alta de 90.89%.
- Existe una disminución más significativa en la etapa relacionada con el ingreso de información de pago, lo que puede representar una oportunidad de mejora en la experiencia del checkout. Se recomienda optimizar este proceso para reducir fricciones y aumentar la probabilidad de finalizar una compra.

# 4. Análisis de Cohortes

Se realiza el análisis de cohortes por primera compra


In [26]:
duckdb.sql("""
WITH first_purchase AS (

    SELECT
        customer_id,
        MIN(order_time) AS first_purchase_date
    FROM orders
    GROUP BY customer_id

)
SELECT * FROM first_purchase
ORDER BY customer_id;
""")

┌─────────────┬─────────────────────┐
│ customer_id │ first_purchase_date │
│    int64    │    timestamp_ns     │
├─────────────┼─────────────────────┤
│           1 │ 2022-03-18 04:16:29 │
│           2 │ 2023-12-16 17:48:30 │
│           3 │ 2020-07-04 07:39:11 │
│           4 │ 2020-09-29 03:07:16 │
│           5 │ 2024-06-15 21:36:57 │
│           6 │ 2021-01-17 20:52:30 │
│           7 │ 2020-09-08 02:27:02 │
│           8 │ 2020-08-08 03:07:31 │
│           9 │ 2024-05-21 13:11:41 │
│          10 │ 2022-12-22 14:22:57 │
│           · │          ·          │
│           · │          ·          │
│           · │          ·          │
│       12307 │ 2023-12-23 06:03:18 │
│       12308 │ 2023-01-18 03:29:45 │
│       12310 │ 2020-09-29 00:42:23 │
│       12311 │ 2020-03-15 22:26:12 │
│       12312 │ 2025-09-17 02:14:13 │
│       12313 │ 2020-04-17 10:42:16 │
│       12317 │ 2021-08-07 00:39:27 │
│       12319 │ 2023-08-12 19:41:32 │
│       12320 │ 2020-09-30 04:37:16 │
│       1232

In [53]:
#Retención por cohortes
duckdb.sql("""
WITH first_purchase AS (

    SELECT
        customer_id,
        MIN(order_time) AS first_purchase_date
    FROM orders
    GROUP BY customer_id

),

customer_activity AS (

    SELECT
        o.customer_id,
        DATE_TRUNC('month', fp.first_purchase_date) AS cohort_month,
        DATE_TRUNC('month', o.order_time) AS order_month
    FROM orders o
    JOIN first_purchase fp
        ON o.customer_id = fp.customer_id
    GROUP BY
        o.customer_id,
        DATE_TRUNC('month', fp.first_purchase_date),
        DATE_TRUNC('month', o.order_time)

),

cohort_data AS (

    SELECT
        customer_id,
        cohort_month,
        order_month,
        DATE_DIFF('month', cohort_month, order_month) AS cohort_index
    FROM customer_activity

),

cohort_size AS (

    SELECT
        cohort_month,
        COUNT(DISTINCT customer_id) AS cohort_size
    FROM cohort_data
    WHERE cohort_index = 0
    GROUP BY cohort_month

),

cohort_counts AS (

    SELECT
        cohort_month,
        cohort_index,
        COUNT(DISTINCT customer_id) AS customers
    FROM cohort_data
    GROUP BY cohort_month, cohort_index

),

cohort_retention AS (

    SELECT
        c.cohort_month,
        c.cohort_index,
        c.customers,
        s.cohort_size,
        ROUND(100.0 * c.customers / s.cohort_size, 2) AS retention_pct
    FROM cohort_counts c
    JOIN cohort_size s
        ON c.cohort_month = s.cohort_month

)

SELECT
    cohort_month,

    ROUND(MAX(CASE WHEN cohort_index = 0 THEN retention_pct END), 2) AS m0,
    ROUND(MAX(CASE WHEN cohort_index = 1 THEN retention_pct END), 2) AS m1,
    ROUND(MAX(CASE WHEN cohort_index = 2 THEN retention_pct END), 2) AS m2,
    ROUND(MAX(CASE WHEN cohort_index = 3 THEN retention_pct END), 2) AS m3,
    ROUND(MAX(CASE WHEN cohort_index = 4 THEN retention_pct END), 2) AS m4,
    ROUND(MAX(CASE WHEN cohort_index = 5 THEN retention_pct END), 2) AS m5,
    ROUND(MAX(CASE WHEN cohort_index = 6 THEN retention_pct END), 2) AS m6,
    ROUND(MAX(CASE WHEN cohort_index = 7 THEN retention_pct END), 2) AS m7,
    ROUND(MAX(CASE WHEN cohort_index = 8 THEN retention_pct END), 2) AS m8,
    ROUND(MAX(CASE WHEN cohort_index = 9 THEN retention_pct END), 2) AS m9,
    ROUND(MAX(CASE WHEN cohort_index = 10 THEN retention_pct END), 2) AS m10,
    ROUND(MAX(CASE WHEN cohort_index = 11 THEN retention_pct END), 2) AS m11,
    ROUND(MAX(CASE WHEN cohort_index = 12 THEN retention_pct END), 2) AS m12

FROM cohort_retention

WHERE cohort_month >= DATE '2025-01-01'
  AND cohort_month < DATE '2026-01-01'
  AND cohort_index <= 12

GROUP BY cohort_month
ORDER BY cohort_month;
""")


┌──────────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┐
│ cohort_month │   m0   │   m1   │   m2   │   m3   │   m4   │   m5   │   m6   │   m7   │   m8   │   m9   │  m10   │  m11   │  m12   │
│     date     │ double │ double │ double │ double │ double │ double │ double │ double │ double │ double │ double │ double │ double │
├──────────────┼────────┼────────┼────────┼────────┼────────┼────────┼────────┼────────┼────────┼────────┼────────┼────────┼────────┤
│ 2025-01-01   │  100.0 │   5.66 │   2.83 │   3.77 │   4.72 │   1.89 │   NULL │   2.83 │   0.94 │   1.89 │   NULL │   NULL │   NULL │
│ 2025-02-01   │  100.0 │   2.27 │   5.68 │   NULL │   7.95 │   3.41 │   4.55 │   5.68 │   3.41 │   NULL │   NULL │   NULL │   NULL │
│ 2025-03-01   │  100.0 │   NULL │   0.87 │   2.61 │   0.87 │   2.61 │   2.61 │   2.61 │   NULL │   NULL │   NULL │   NULL │   NULL │
│ 2025-04-01   │  100.0 │    2.8 │   1.87 │   0.93 │   5.61 │ 

#### Comentario
- La cohorte con mejor retención en su primer mes fue la de enero (2025-01-01) con apenas un 5.66%, mientras que la peor fue la de junio (2025-06-01) con un alarmante 1.05%.
- La cohorte de febrero (2025-02-01) muestra un comportamiento muy inusual. Comienza en m1 con un bajo 2.27%, pero luego sube en m2 (5.68%) y alcanza su punto máximo en m4 con un 7.95%. Esto sugiere que en el mes 4 de esa cohorte (que correspondería a junio de 2025) se lanzó alguna campaña de reactivación muy efectiva, una actualización importante del servicio, o un incentivo que hizo regresar a usuarios que ya se habían ido.
- Se observa que la capacidad de la plataforma para retener nuevos usuarios se fue deteriorando a lo largo de 2025. esto dado que las cohortes más recientes son menos leales que las primeras del año.

# Análisis de Funnel de Conversión y Cohortes

##Contexto y Objetivo

Este análisis evalúa el comportamiento de los usuarios durante el proceso de compra y su permanencia en el tiempo dentro de una plataforma de comercio electrónico. Para ello se emplearon dos enfoques complementarios: un Análisis de Funnel, orientado a medir la eficiencia del proceso de conversión, y un Análisis de Cohortes, enfocado en evaluar la retención de usuarios según el mes de su primera compra.

El objetivo principal es identificar oportunidades de mejora tanto en el proceso de compra como en la fidelización de clientes, permitiendo establecer estrategias que incrementen la conversión, reduzcan el abandono y mejoren el valor del cliente a largo plazo.

*Cobertura de los datos:* El análisis comprende las interacciones registradas entre los años 2020 y e incluye los eventos de visualización de productos, adición al carrito, inicio del proceso de pago (checkout) y compra final. Adicionalmente, Se analizaron las cohortes de usuarios en el año 2025 clasificadas según el mes de su primera compra para evaluar su comportamiento de retención durante los meses posteriores.

##Metodología

Inicialmente se realizó la limpieza y validación de los registros para asegurar la consistencia de los eventos del proceso de compra. Posteriormente se estructuró el recorrido de cada usuario dentro del embudo de conversión, identificando las etapas de visualización del producto, adición al carrito, inicio del checkout y compra.

Se calcularon las tasas de conversión entre cada una de las etapas del embudo:

Conversión Página → Carrito.
Conversión Carrito → Checkout.
Conversión Checkout → Compra.

En paralelo, se construyó una matriz de cohortes agrupando a los usuarios según el mes de su primera compra. Para cada cohorte se calculó el porcentaje de usuarios que regresaron durante los meses posteriores (m1, m2, m3, etc.), permitiendo evaluar la evolución de la retención y detectar patrones de fidelización o abandono.

Finalmente, se compararon los resultados obtenidos para identificar tanto los puntos críticos del proceso de compra como el comportamiento de las cohortes a lo largo del tiempo.

##Hallazgos Iniciales
### Funnel
- El embudo de conversión evidencia un proceso de compra altamente eficiente. De los 19.945 usuarios que visualizaron un producto, 19.660 lo añadieron al carrito, alcanzando una conversión del 98,57%. Posteriormente, 17.898 usuarios iniciaron el proceso de pago, lo que representa una conversión del 91,04% desde el carrito hacia el checkout.

- 16.268 usuarios completaron la compra, obteniendo una conversión del 90,89% entre el checkout y la compra final. Estos resultados indican que, una vez el usuario inicia el proceso de compra, existe una elevada probabilidad de finalizar la transacción.

- La mayor disminución de usuarios se presenta durante el proceso de checkout, lo que sugiere que factores como el ingreso de información de pago, costos adicionales, tiempos de carga o complejidad del proceso podrían generar abandono antes de completar la compra.

### Cohortes

- El análisis de retención muestra un comportamiento esperado en plataformas de comercio electrónico: la mayor pérdida de usuarios ocurre durante el primer mes posterior a la compra.

- La cohorte correspondiente a enero de 2025 presenta la mejor retención inicial, con un 5,66% de usuarios que regresan durante el primer mes, mientras que la cohorte de junio de 2025 registra la menor retención, con apenas un 1,05%.

- Se observa un comportamiento particularmente interesante en la cohorte de febrero de 2025, cuya retención aumenta progresivamente desde 2,27% en el primer mes hasta alcanzar un máximo de 7,95% en el cuarto mes. Este patrón podría estar asociado con campañas de remarketing, promociones estacionales, programas de fidelización o lanzamientos de nuevos productos que incentivaron el regreso de clientes previamente inactivos.

- En términos generales, las cohortes más recientes presentan menores niveles de retención durante los primeros meses de seguimiento, lo que sugiere una disminución en la capacidad de fidelizar nuevos clientes o un tiempo de observación aún insuficiente para evaluar completamente su comportamiento.

## Recomendaciones
- Mantener las buenas prácticas implementadas en las etapas iniciales del embudo, dado que presentan tasas de conversión superiores al 90%.
- Analizar detalladamente el proceso de checkout para identificar posibles causas de abandono relacionadas con métodos de pago, tiempos de respuesta, costos de envío o complejidad del formulario.
- Implementar pruebas A/B sobre el proceso de pago para reducir la fricción y aumentar la conversión final.
- Diseñar campañas de remarketing dirigidas a usuarios que abandonan el checkout antes de finalizar la compra.
- Replicar las estrategias comerciales implementadas durante la cohorte de febrero, ya que muestran un comportamiento de reactivación superior al resto de cohortes.
- Fortalecer las estrategias de fidelización mediante programas de recompensas, recomendaciones personalizadas y campañas de correo electrónico orientadas al regreso de clientes.
- Realizar un seguimiento continuo de las cohortes más recientes para determinar si la disminución observada responde a cambios en el comportamiento de los usuarios o simplemente a un menor tiempo de maduración de las cohortes.

# Resumen Ejecutivo
##📊 Rendimiento del Funnel
- Usuarios que visualizaron productos: 19.945
- Usuarios que añadieron productos al carrito: 19.660
- suarios que iniciaron el checkout: 17.898
- Usuarios que completaron la compra: 16.268
- Conversión Página → Carrito: 98,57%
- Conversión Carrito → Checkout: 91,04%
- Conversión Checkout → Compra: 90,89%

##📈 Hallazgos de Cohortes
- La mejor retención en el primer mes corresponde a la cohorte de enero de 2025 (5,66%).
- La cohorte de junio de 2025 presenta la menor retención inicial (1,05%).
- La cohorte de febrero de 2025 evidencia un comportamiento atípico, alcanzando una retención máxima de 7,95% en el cuarto mes, lo que podría reflejar el impacto positivo de acciones de reactivación o fidelización.
- En general, la retención disminuye considerablemente después del primer mes, indicando que la fidelización continúa siendo uno de los principales desafíos del negocio.

##💡 Recomendaciones Estratégicas
- Optimizar el proceso de checkout para disminuir el abandono antes de la compra.
- Analizar los factores que favorecieron el comportamiento excepcional de la cohorte de febrero y replicar esas estrategias.
- Implementar campañas automatizadas de retención y reactivación durante los primeros meses posteriores a la compra.
- Desarrollar programas de fidelización que incentiven compras recurrentes y aumenten el valor del cliente a largo plazo.
- Monitorear periódicamente los indicadores del funnel y las cohortes para evaluar el impacto de las mejoras implementadas y facilitar la toma de decisiones basada en datos.